In [9]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# record utils
from lerobot.record import record, RecordConfig, DatasetRecordConfig

# torch
from torch import cuda

# utils
import pprint
from dotenv import load_dotenv

# my code
from robot.robot_config import robot_config, robot_ext_config
from robot.robot_const import FOLDED_START_POSE
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.utils import check_resume

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Set Params:

In [10]:
PUSH_TO_HUB       = True
SAVE_TO_DATASET   = True
REPO_NAME         = 'so101_car_pick_and_place'
EXPERIMENT_NAME   = '96_episodes_v0'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE       = 'act'
NUM_EPISODES      = 8
FPS               = 30
EPISODE_TIME_SEC  = 30
RESET_TIME_SEC    = 5
EVAL_TYPE         = 'real_v0'
USE_EXT_ROBOT = False  # robot config !!

Log to HF if pushing:

In [11]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `pc` has been saved to /home/jonathan/.cache/huggingface/stored_tokens
Your token has been saved to /home/jonathan/.cache/huggingface/token
Login successful.
The current active token is: `pc`


In [12]:
# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    # load from Hugging Face Hub
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config)

ACTConfig(n_obs_steps=1,
          input_features={'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                                      shape=(3,
                                                                             480,
                                                                             640)),
                          'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                                        shape=(3,
                                                                               480,
                                                                               640)),
                          'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                                             shape=(6,))},
          output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>,
  

In [13]:
policy_config.input_features

{'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(6,)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640))}

Policy Overrides:

In [14]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
policy_config.n_action_steps = 25
policy_config.temporal_ensemble_coeff = None
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 25, temporal_ensemble = None


Build Dataset:

In [15]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_ext_config if USE_EXT_ROBOT else robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [16]:
rc.resume = check_resume(dataset_path)
record(rc, 
    save_to_ds    = SAVE_TO_DATASET,
    reset_pose    = FOLDED_START_POSE,
    give_feedback = False,
    log_to_file   = False)

INFO 2026-01-31 16:03:58 t/record.py:434 {'dataset': {'episode_time_s': 30,
             'fps': 30,
             'num_episodes': 8,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': True,
             'push_to_hub': True,
             'rename_map': {},
             'repo_id': 'jonathm126/eval_so101_car_pick_and_place_96_episodes_v0_real_v0',
             'reset_time_s': 5,
             'root': '/home/jonathan/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_car_pick_and_place/96_episodes_v0-real_v0',
             'single_task': 'eval dataset for so101_car_pick_and_place with '
                            'policy 96_episodes_v0, mode = real_v0',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'cuda',
            'dim_feedforward': 3200,
            'dim_model': 512,


Loading weights from local directory


INFO 2026-01-31 16:04:01 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2026-01-31 16:04:03 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2026-01-31 16:04:03 follower.py:104 so_101_follower SO101Follower connected.
INFO 2026-01-31 16:04:04 ls/utils.py:227 Recording episode 89
INFO 2026-01-31 16:04:34 ls/utils.py:227 Reset the environment
Map: 100%|██████████| 858/858 [00:00<00:00, 3387.57 examples/s]
Generating train split: 67759 examples [00:00, 4100918.37 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : u

Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 751/751 [00:00<00:00, 3141.60 examples/s]
Generating train split: 68510 examples [00:00, 4467464.23 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-fr

...

INFO 2026-01-31 16:07:43 ls/utils.py:227 Re-record episode
INFO 2026-01-31 16:07:43 ls/utils.py:227 Recording episode 93
INFO 2026-01-31 16:08:06 ls/utils.py:227 Reset the environment


Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 680/680 [00:00<00:00, 3257.59 examples/s]
Generating train split: 70909 examples [00:00, 4519349.97 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-fr

ConnectionError: Failed to sync read 'Present_Position' on ids=[1, 2, 3, 4, 5, 6] after 1 tries. [TxRxResult] There is no status packet!

Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Escape key pressed. Stopping data recording...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...
Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pres